# Figure 5 — XBP1 Stress-Induced Defense Consolidation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aistanbulresearch/msep/blob/main/notebooks/figures/figure5_xbp1.ipynb)

Reproduces **Figure 5** from Çavuş & Kuşkucu (2026): XBP1-high CSC tighten coordination across all three defence pathways simultaneously (stress-induced consolidation), a pattern that extends to five of nine cancer types tested.

**Panels**

| Panel | What it shows | Source |
|---|---|---|
| A | Distribution of XBP1 expression in CSC, 85th-percentile threshold annotated | `result.xbp1` + histogram |
| B | Pathway CV: XBP1-zero vs XBP1-high, with ΔCV and permutation p-values | `result.xbp1` + `result.consolidation_score()` |
| C | Pan-cancer consolidation scorecard (9 cancer types × 3 pathways) | WP-3 Census output |

## 1. Install

In [ ]:
!pip install -q --upgrade-strategy only-if-needed 'msep[plotting]>=1.1.0'

## 2. Load and run XBP1 consolidation

In [ ]:
import msep
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

adata = msep.datasets.load_example(n_cells=500, seed=42)
result = msep.profile(
    adata,
    pathways='cancer_defense',
    cell_type_key='cell_type',
    compute_bootstrap=False,
    compute_gene_cv=False,
    compute_xbp1=True,
    perturbation_cell_type='CSC_TBXT+',
)
print('XBP1 group CVs:')
print(result.xbp1.round(3))
print('\nConsolidation score:')
print(result.consolidation_score().round(3))

## 3. Panel A — XBP1 expression distribution

In [ ]:
csc_mask = adata.obs['cell_type'] == 'CSC_TBXT+'
xbp1_idx = adata.var_names.get_loc('XBP1')
xbp1_counts = adata.X.toarray()[csc_mask, xbp1_idx]
nonzero = xbp1_counts[xbp1_counts > 0]
threshold = np.percentile(nonzero, 85) if len(nonzero) else 0
frac_nonzero = (xbp1_counts > 0).mean()
frac_high = (xbp1_counts >= threshold).mean() if threshold > 0 else 0

fig_a, ax_a = plt.subplots(figsize=(6, 4))
ax_a.hist(nonzero, bins=25, color='#9B59B6', edgecolor='white')
if threshold > 0:
    ax_a.axvline(threshold, ls='--', color='red',
                 label=f'85th pct threshold = {threshold:.1f}')
    ax_a.legend()
ax_a.set_xlabel('XBP1 raw count (nonzero cells)')
ax_a.set_ylabel('Cells')
ax_a.set_title(f'A · XBP1 distribution in CSC — {frac_nonzero:.1%} nonzero, {frac_high:.1%} high')
ax_a.spines[['top', 'right']].set_visible(False)
fig_a.tight_layout()
fig_a.savefig('figure5_panelA_xbp1_dist.pdf', bbox_inches='tight')
fig_a.savefig('figure5_panelA_xbp1_dist.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Panel B — Pathway CV by XBP1 group

In [ ]:
xbp1 = result.xbp1.copy()
pivot = xbp1.pivot(index='pathway', columns='xbp1_group', values='cv').reindex(
    ['emt', 'ferroptosis', 'immune_evasion', 'housekeeping']
)
pivot = pivot[['XBP1-zero', 'XBP1-low', 'XBP1-high']]

fig_b, ax_b = plt.subplots(figsize=(7, 4.5))
pivot.plot(kind='bar', ax=ax_b, color=['#BDC3C7', '#9B59B6', '#8E44AD'], edgecolor='black', linewidth=0.5)
ax_b.set_ylabel('Across-cells CV')
ax_b.set_title('B · Pathway CV by XBP1 group (CSC only)')
ax_b.tick_params(axis='x', rotation=0)
ax_b.legend(title='XBP1 group', loc='upper right')
ax_b.spines[['top', 'right']].set_visible(False)

consol = result.consolidation_score()
consol_map = consol.set_index('pathway')
for i, pw in enumerate(pivot.index):
    if pw in consol_map.index:
        d = consol_map.loc[pw, 'delta']
        ax_b.text(i, pivot.loc[pw].max() + 0.05,
                  f'Δ = {d:+.2f}', ha='center', fontsize=9,
                  color='green' if d < 0 else 'red')
fig_b.tight_layout()
fig_b.savefig('figure5_panelB_xbp1_groups.pdf', bbox_inches='tight')
fig_b.savefig('figure5_panelB_xbp1_groups.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Panel C — Pan-cancer consolidation scorecard

Reproduces the paper's 9-cancer scorecard from published values. When the WP-3 Census pipeline lands, regenerate this matrix from live data (expanded to 20 cancer types).

In [ ]:
import seaborn as sns

# Paper's Table S8 — 1 = consolidated (p < 0.05), 0 = not
scorecard = pd.DataFrame({
    'ferroptosis':    [1, 1, 1, 1, 1, 1, 0, 1, 0],
    'immune_evasion': [1, 1, 1, 1, 1, 0, 1, 1, 1],
    'emt':            [1, 1, 1, 1, 1, 1, 0, 0, 0],
}, index=[
    'chordoma', 'lung_adenocarcinoma', 'triple_negative_breast',
    'nonpapillary_renal', 'uveal_melanoma', 'glioblastoma',
    'breast_ductal', 'squamous_lung', 'basal_cell_carcinoma',
])
scorecard['total'] = scorecard.sum(axis=1)

fig_c, ax_c = plt.subplots(figsize=(6, 4.5))
sns.heatmap(scorecard.drop(columns='total'), annot=True, fmt='d',
            cmap='Purples', cbar=False, linewidths=0.5, linecolor='white',
            ax=ax_c)
for i, v in enumerate(scorecard['total']):
    ax_c.text(3.3, i + 0.5, f'{v}/3', va='center', fontsize=10,
              fontweight='bold', color='#8E44AD')
ax_c.set_title('C · Pan-cancer XBP1 consolidation scorecard\n(5/9 cancer types show full consolidation)')
ax_c.set_xlabel('')
ax_c.set_ylabel('')
fig_c.tight_layout()
fig_c.savefig('figure5_panelC_scorecard.pdf', bbox_inches='tight')
fig_c.savefig('figure5_panelC_scorecard.png', dpi=300, bbox_inches='tight')
plt.show()